# Figure 5 mechanism and robustness supplement | Supplementary Fig. S6

This notebook extends the Figure 5 analyses: Extended Data Fig. 7a–h evaluates gradient and functional-importance mechanisms, Extended Data Fig. 7i–l tests FC peak-timing robustness, and Extended Data Fig. 7m–n examines FC-guided residual stability across datasets.

In [ ]:
from pathlib import Path
import json
import platform
import sys

import matplotlib.pyplot as plt
import numpy as np
import scipy
import torch
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms


def locate_code_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for base in candidates:
        if base.name == 'Supplementary_fig_code' and (base / 'utils' / 'fig5_mechanism.py').exists():
            return base
        candidate = base / 'FC-IS_code' / 'Supplementary_fig_code'
        if (candidate / 'utils' / 'fig5_mechanism.py').exists():
            return candidate
    raise FileNotFoundError('Cannot locate FC-IS_code/Supplementary_fig_code.')


CODE_DIR = locate_code_dir()
FC_IS_ROOT = CODE_DIR.parent
MLP_ROOT = FC_IS_ROOT / 'MLP'
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from utils.fig5_mechanism import (
    Fig5SupplementConfig,
    collect_balanced_disjoint_sets,
    export_figure_bundle,
    plot_fig5a_f_supplement,
    run_or_load_all_seeds,
    statistical_summary,
)

print({'python': platform.python_version(), 'torch': torch.__version__,
       'numpy': np.__version__, 'scipy': scipy.__version__})
from utils.fig5_robustness import (
    Fig5GHConfig, Fig5IConfig, export_figure_bundle as export_robustness_bundle,
    gh_peak_steps, i_performance_effects, plot_fig5gh_robustness,
    plot_fig5i_stability, run_or_load_gh, run_or_load_i,
    summarize_gh_results, summarize_i_results,
)
MLP_DIR = MLP_ROOT
print('Code directory:', CODE_DIR)

## S6a-h | Early FC identifies gradient-rich and functionally important units

In [ ]:
DATA_ROOT = MLP_ROOT / 'data'
MECHANISM_RESULT_DIR = CODE_DIR / 'results' / 'fig5_FC_mechanism_supp'
MECHANISM_OUTPUT_DIR = CODE_DIR / 'outputs' / 'fig5_FC_mechanism_supp'
MECHANISM_RESULT_DIR.mkdir(parents=True, exist_ok=True)
MECHANISM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

mechanism_config = Fig5SupplementConfig(
    seeds=tuple(range(15)),       # independent training runs; keep >= 15
    learning_rate=0.05,           # same as the main Figure 5 notebook
    max_steps=300,
    selection_step=180,           # resolves the original 150/180 inconsistency
    snapshot_interval=10,
    selection_fraction=0.20,
    continuation_steps=120,
    random_mask_repeats=20,
)
mechanism_config.validate()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OVERWRITE = False
NUM_WORKERS = 2
print('Device:', DEVICE)
print('Data root:', DATA_ROOT)
print('Number of independent seeds:', len(mechanism_config.seeds))

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

try:
    train_dataset = datasets.MNIST(DATA_ROOT, train=True, download=False, transform=transform)
    test_dataset = datasets.MNIST(DATA_ROOT, train=False, download=False, transform=transform)
except RuntimeError as exc:
    raise RuntimeError(
        f'MNIST was not found under {DATA_ROOT}. Place the same MNIST files used by the main MLP analysis there.'
    ) from exc

selection_inputs, selection_labels, outcome_inputs, outcome_labels = (
    collect_balanced_disjoint_sets(test_dataset, samples_per_class=50, num_classes=10)
)
assert len(selection_inputs) == len(outcome_inputs) == 500
assert not torch.equal(selection_inputs, outcome_inputs)

def make_train_loader(seed):
    generator = torch.Generator()
    generator.manual_seed(int(seed))
    return DataLoader(
        train_dataset,
        batch_size=256,
        shuffle=True,
        generator=generator,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=False,
        drop_last=False,
    )

print('Selection set A:', tuple(selection_inputs.shape))
print('Outcome set B:', tuple(outcome_inputs.shape))

In [ ]:
mechanism_results = run_or_load_all_seeds(
    train_loader_factory=make_train_loader,
    selection_inputs=selection_inputs,
    outcome_inputs=outcome_inputs,
    outcome_labels=outcome_labels,
    config=mechanism_config,
    result_dir=MECHANISM_RESULT_DIR,
    device=DEVICE,
    overwrite=OVERWRITE,
)
print('Completed seeds:', mechanism_results['seed'].astype(int).tolist())
print('Combined source data:', MECHANISM_RESULT_DIR / 'fig5a_f_all_seeds.npz')

In [ ]:
assert mechanism_results['seed'].shape[0] >= 15
assert np.unique(mechanism_results['seed']).size == mechanism_results['seed'].size
assert mechanism_results['group_gradient'].shape[1:] == (3, mechanism_config.max_steps)
assert mechanism_results['acute_loss'].shape[1:] == (4, len(mechanism_config.masking_fractions))
assert mechanism_results['persistent_loss'].shape[1] == 5
summary = statistical_summary(mechanism_results)
print(json.dumps(summary, indent=2, ensure_ascii=False))

In [ ]:
mechanism_fig = plot_fig5a_f_supplement(mechanism_results, mechanism_config)
paths = export_figure_bundle(
    mechanism_fig, mechanism_results, MECHANISM_OUTPUT_DIR, stem='fig5_FC_mechanism_supp'
)
plt.show()
for kind, path in paths.items():
    print(f'{kind:>12}: {path}')

## S6i-l | Adam/BatchNorm advance FC peak timing across definitions and datasets

In [ ]:
DATA_ROOT = MLP_DIR / 'data'
RESULT_ROOT = CODE_DIR / 'results' / 'fig5_FC_peak_timing_supp'
OUTPUT_ROOT = CODE_DIR / 'outputs' / 'fig5_FC_peak_timing_supp'
DATASET_ORDER = ('MNIST', 'FashionMNIST', 'KMNIST')
TOP_FC_FRACTIONS = (0.025, 0.05, 0.10, 0.20, 0.40)
NUM_WORKERS = 2
OVERWRITE = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

gh_config = Fig5GHConfig(
    seeds=tuple(range(20)), hidden_dims=(300, 300), batch_size=256,
    total_steps=300, evaluation_interval=5, learning_rate=0.05,
    adam_learning_rate=1e-3, top_fc_fractions=TOP_FC_FRACTIONS,
    smoothing_window=5, peak_tolerance=0.01,
    analysis_samples=2000,
)
gh_config.validate()
print('Device:', DEVICE, '| seeds:', len(gh_config.seeds), '| top-FC fractions:', TOP_FC_FRACTIONS)

In [ ]:
DATASET_SPECS = {
    'MNIST': (torchvision.datasets.MNIST, (0.1307,), (0.3081,)),
    'FashionMNIST': (torchvision.datasets.FashionMNIST, (0.2860,), (0.3530,)),
    'KMNIST': (torchvision.datasets.KMNIST, (0.1904,), (0.3475,)),
}

def load_dataset(name):
    dataset_class, mean, std = DATASET_SPECS[name]
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])
    try:
        train_set = dataset_class(DATA_ROOT, train=True, transform=transform, download=False)
        test_set = dataset_class(DATA_ROOT, train=False, transform=transform, download=False)
    except RuntimeError as exc:
        raise RuntimeError(f'{name} was not found under {DATA_ROOT}.') from exc
    loader = DataLoader(test_set, batch_size=gh_config.analysis_samples, shuffle=False, num_workers=0)
    analysis_inputs, _ = next(iter(loader))
    return train_set, analysis_inputs

gh_datasets = {name: load_dataset(name) for name in DATASET_ORDER}
for name, (train_set, inputs) in gh_datasets.items():
    print(name, len(train_set), tuple(inputs.shape))

In [ ]:
gh_results = {'adam': {}, 'batchnorm': {}}
for experiment in gh_results:
    for dataset_name in DATASET_ORDER:
        train_set, analysis_inputs = gh_datasets[dataset_name]
        print(f'Running {experiment} on {dataset_name}...')
        gh_results[experiment][dataset_name] = run_or_load_gh(
            experiment=experiment, dataset_name=dataset_name,
            train_dataset=train_set, analysis_inputs=analysis_inputs,
            config=gh_config, result_root=RESULT_ROOT, device=DEVICE,
            num_workers=NUM_WORKERS, overwrite=OVERWRITE,
        )

In [ ]:
gh_statistics = summarize_gh_results(gh_results, gh_config)
for experiment, dataset_stats in gh_statistics.items():
    print('\n', experiment)
    for dataset_name, values in dataset_stats.items():
        print(dataset_name, values)

gh_fig = plot_fig5gh_robustness(gh_results, gh_config, DATASET_ORDER)
gh_source_data = {'top_fc_fractions': np.asarray(gh_config.top_fc_fractions)}
for experiment, dataset_results in gh_results.items():
    for dataset_name, result in dataset_results.items():
        prefix = f'{experiment}_{dataset_name}'
        gh_source_data[f'{prefix}_seeds'] = result['seeds']
        gh_source_data[f'{prefix}_steps'] = result['steps']
        gh_source_data[f'{prefix}_fc_curves'] = result['fc_curves']
        gh_source_data[f'{prefix}_peak_steps'] = gh_peak_steps(result, gh_config)

gh_paths = export_robustness_bundle(
    gh_fig, OUTPUT_ROOT, 'fig5_FC_peak_timing_supp',
    source_data=gh_source_data, statistics=gh_statistics,
)
plt.show()
for kind, path in gh_paths.items():
    print(kind, path)

## S6m-n | FC-guided residual stability across datasets

In [ ]:
DATA_ROOT = MLP_DIR / 'data'
RESULT_ROOT = CODE_DIR / 'results' / 'fig5_FC_guided_stability_supp'
OUTPUT_ROOT = CODE_DIR / 'outputs' / 'fig5_FC_guided_stability_supp'
DATASET_ORDER = ('MNIST', 'FashionMNIST', 'KMNIST')
NUM_WORKERS = 2
OVERWRITE = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

FC_GATE_STRENGTH = 3.0
FC_GATE_TOP_FRACTION = 0.5
FC_COUPLING_STRENGTH = 0.80
FC_COUPLING_STEPS = 40

i_config = Fig5IConfig(
    seeds=tuple(range(20)), hidden_dim=256, batch_size=512,
    total_steps=300, evaluation_interval=5, learning_rate=0.05,
    analysis_samples=2000, top_fc_ratio=0.05, fc_threshold=0.70,
    gate_strength=FC_GATE_STRENGTH,
    gate_top_fraction=FC_GATE_TOP_FRACTION,
    fc_coupling_strength=FC_COUPLING_STRENGTH,
    coupling_steps=FC_COUPLING_STEPS,
)
i_config.validate()
print('Device:', DEVICE, '| seeds:', len(i_config.seeds), '| FC parameters:', {
    'gate_strength': i_config.gate_strength,
    'gate_top_fraction': i_config.gate_top_fraction,
    'fc_coupling_strength': i_config.fc_coupling_strength,
    'coupling_steps': i_config.coupling_steps,
})

In [ ]:
DATASET_SPECS = {
    'MNIST': (torchvision.datasets.MNIST, (0.1307,), (0.3081,)),
    'FashionMNIST': (torchvision.datasets.FashionMNIST, (0.2860,), (0.3530,)),
    'KMNIST': (torchvision.datasets.KMNIST, (0.1904,), (0.3475,)),
}

def load_dataset(name):
    dataset_class, mean, std = DATASET_SPECS[name]
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])
    try:
        train_set = dataset_class(DATA_ROOT, train=True, transform=transform, download=False)
        test_set = dataset_class(DATA_ROOT, train=False, transform=transform, download=False)
    except RuntimeError as exc:
        raise RuntimeError(f'{name} was not found under {DATA_ROOT}.') from exc
    loader = DataLoader(test_set, batch_size=i_config.analysis_samples, shuffle=False, num_workers=0)
    analysis_inputs, analysis_targets = next(iter(loader))
    return train_set, analysis_inputs, analysis_targets

i_datasets = {name: load_dataset(name) for name in DATASET_ORDER}
for name, (train_set, inputs, _) in i_datasets.items():
    print(name, len(train_set), tuple(inputs.shape))

In [ ]:
dataset_results = {}
for dataset_name in DATASET_ORDER:
    train_set, analysis_inputs, analysis_targets = i_datasets[dataset_name]
    print('Running FC-guided stability:', dataset_name)
    dataset_results[dataset_name] = run_or_load_i(
        dataset_name=dataset_name, train_dataset=train_set,
        analysis_inputs=analysis_inputs, analysis_targets=analysis_targets,
        config=i_config, result_root=RESULT_ROOT, device=DEVICE,
        num_workers=NUM_WORKERS, overwrite=OVERWRITE,
    )


In [ ]:
i_statistics = summarize_i_results(dataset_results)
for dataset_name, row in i_statistics.items():
    print(dataset_name, row)

i_fig = plot_fig5i_stability(dataset_results, i_config, DATASET_ORDER)
i_source_data = {
    'gate_strength': np.asarray(i_config.gate_strength),
    'gate_top_fraction': np.asarray(i_config.gate_top_fraction),
    'fc_coupling_strength': np.asarray(i_config.fc_coupling_strength),
    'coupling_steps': np.asarray(i_config.coupling_steps),
}
for dataset_name, result in dataset_results.items():
    i_source_data[f'{dataset_name}_seeds'] = result['seeds']
    i_source_data[f'{dataset_name}_steps'] = result['steps']
    i_source_data[f'{dataset_name}_curves'] = result['curves']
    effects = i_performance_effects(result)
    i_source_data[f'{dataset_name}_loss_auc_benefit'] = effects['loss_auc_benefit']
    i_source_data[f'{dataset_name}_accuracy_auc_benefit'] = effects['accuracy_auc_benefit']

i_paths = export_robustness_bundle(
    i_fig, OUTPUT_ROOT, 'fig5_FC_guided_stability_supp',
    source_data=i_source_data, statistics=i_statistics,
)
plt.show()
for kind, path in i_paths.items():
    print(kind, path)


## Reporting

- S6a-h: n = 15 independently initialized and trained models.
- S6i-n: n = 20 paired seeds per condition and dataset.
- Error bars are 95% confidence intervals across seeds; neurons and FC edges are not inferential replicates.